# Unrolled-ADMM demo

## Set some constants

In [1]:
REPO = "Vdv09/Unrolled-ADMM"
WORKDIR = "/content/Unrolled-ADMM"

DATASET_ZIP_URL = "https://drive.google.com/file/d/1Uak6xCdh_iNNZ58sikzIFZZQOSMp8203/view?usp=drive_link"  # CustomDirDataset, this is an example

CHECKPOINTS_FOLDER_URL = "https://drive.google.com/drive/folders/1jhGziJtGEc1pitPdfIyvdhp_Xt9NuH2C"

# leadmm5-pre_post | leadmm5-pre | leadmm5-post | admm20
MODEL_PRESET = "leadmm5-pre_post"

PRESETS = {
    "leadmm5-pre_post": {
        "model": "modular_leadmm5",
        "variant": "pre_post",
        "checkpoint": "checkpoints/leadmm5-pre_post/model_best.pth",
    },
    "leadmm5-pre": {
        "model": "modular_leadmm5",
        "variant": "pre",
        "checkpoint": "checkpoints/leadmm5-pre/model_best.pth",
    },
    "leadmm5-post": {
        "model": "modular_leadmm5",
        "variant": "post",
        "checkpoint": "checkpoints/leadmm5-post/model_best.pth",
    },
    "admm20": {
        "model": "admm20",
        "variant": None,
        "checkpoint": "checkpoints/admm20/model_best.pth",
    },
}

SAVE_PATH = "testing_repo"

CURRENT_CONFIG = PRESETS[MODEL_PRESET]

## Clone repository

In [2]:
!git clone https://github.com/{REPO}.git {WORKDIR}
%cd {WORKDIR}

fatal: could not create leading directories of '/content/Unrolled-ADMM': Read-only file system
[Errno 2] No such file or directory: '/content/Unrolled-ADMM'
/Users/mihailbugrysev/HSE/Homework/2_course/DL/Project/Unrolled-ADMM


## Install needed dependencies

In [ ]:
!pip install -q hydra-core omegaconf datasets pillow tqdm scipy pyffs torchmetrics opencv-python gdown
!pip install -q git+https://github.com/ebezzam/waveprop.git --no-deps
!pip install -q git+https://github.com/ebezzam/slm-controller.git --no-deps
!pip install -q git+https://github.com/pvigier/perlin-numpy.git@5e26837db14042e51166ebcad4c0df2c1907016

## Download checkpoints from Google Drive

In [ ]:
from pathlib import Path
import gdown

Path("checkpoints").mkdir(exist_ok=True)

gdown.download_folder(CHECKPOINTS_FOLDER_URL, output="checkpoints")

## Download compressed custom dataset (you can see more in src.datasets.custom_dir_dataset)

In [ ]:
import zipfile
import shutil

zip_path = Path("/content/custom_dataset.zip")
gdown.download(DATASET_ZIP_URL, str(zip_path), fuzzy=True)

extract_root = Path("/content/custom_extract")
extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_root)

lensless_dirs = [p for p in extract_root.rglob("lensless") if p.is_dir()]
CUSTOM_DATA_DIR = str(lensless_dirs[0].parent)
print("Dataset dir:", CUSTOM_DATA_DIR)

## Run inference

In [ ]:
cmd = f"""python inference.py \
  datasets=custom_dir \
  datasets.inference.data_dir={CUSTOM_DATA_DIR} \
  model={CURRENT_CONFIG['model']} \
  {f"model.variant={CURRENT_CONFIG['variant']}" if CURRENT_CONFIG["variant"] else ""} \
  inferencer.from_pretrained={CURRENT_CONFIG['checkpoint']} \
  inferencer.save_path={SAVE_PATH}"""

!{cmd}

## Visualize samples and calculating metrics

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image


pred_dir = Path("data/saved") / SAVE_PATH / "inference"
custom_dir = Path(CUSTOM_DATA_DIR)

ids = sorted(p.stem for p in pred_dir.glob("*.png"))
sample_ids = random.sample(ids, min(3, len(ids)))

for image_id in sample_ids:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(Image.open(custom_dir / "lensless" / f"{image_id}.png"))
    axes[0].set_title("lensless")

    axes[1].imshow(Image.open(pred_dir / f"{image_id}.png"))
    axes[1].set_title("reconstruction")

    real_path = custom_dir / "lensed" / f"{image_id}.png"

    if real_path.exists():
        axes[2].imshow(Image.open(real_path))
        axes[2].set_title("real image")
    else:
        axes[2].text(0.5, 0.5, "no real image", ha="center")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(image_id)
    plt.show()

And what about calculating metrics (apparently - you must have directory with real lensed images):

In [ ]:
pred_dir = Path("data/saved") / SAVE_PATH / "inference"

!python calculate_metrics.py --pred-dir {pred_dir} --dataset-dir {CUSTOM_DATA_DIR} --device cuda

## Speed benchmark (for all models)

In [ ]:
import re
import subprocess
from pathlib import Path

SPEED_METHODS = [
    ("ADMM-100", ["--model", "admm100"]),
    ("ADMM-20", ["--model", "admm20", "--checkpoint", "checkpoints/admm20/model_best.pth"]),
    ("LeADMM-5 pre+post", ["--model", "modular_leadmm5", "--variant", "pre_post", "--checkpoint", "checkpoints/leadmm5-pre_post/model_best.pth"]),
    ("LeADMM-5 pre", ["--model", "modular_leadmm5", "--variant", "pre", "--checkpoint", "checkpoints/leadmm5-pre/model_best.pth"]),
    ("LeADMM-5 post", ["--model", "modular_leadmm5", "--variant", "post", "--checkpoint", "checkpoints/leadmm5-post/model_best.pth"]),
]

common = ["--device", "cuda", "--batch-size", "1", "--warmup", "3", "--repeats", "20"]

for name, args in SPEED_METHODS:
    if name != "ADMM-100":
        checkpoint = Path(args[args.index("--checkpoint") + 1])

    cmd = ["python", "scripts/check_speed.py", *args, *common]
    proc = subprocess.run(cmd, capture_output=True, text=True)

    m = re.search(r"Speed:\s+([\d.]+)\s+images/sec", proc.stdout)

    images_per_second = float(m.group(1))
    print(f"{name} - {images_per_second} images/sec")